# Analisi repository di Github più popolari
## Caricamento file dataset
Carico i vari file csv dei domini dei dataset tramite la libreria Pandas

In [ ]:
import pandas
import os
from ipywidgets import interact, widgets

datasets = []

for domain in os.listdir("../data/domains"):
    datasets.append((domain.split(".")[0], pandas.read_csv(os.path.join("../data/domains", domain))))

full_dataset = pandas.read_csv("../data/github_top_repositories.csv")

print("Lista di dataset: " + str([x[0] for x in datasets]))

### Esempio di come è strutturato un dominio del dataset

In [ ]:
datasets[0][1]

## Analisi del dataset
Importo matplotlib e numpy in quanto sono librerie che verranno utilizzate in seguito

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

### Ambiti più popolari
Possiamo confrontare la popolarità tra i vari dominii, osservando il numero di stelle per ognuno di essi

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(25, 6))

for (domain, dataset) in datasets:
    ax[0].bar(domain, len(dataset))
    ax[0].tick_params(axis='x', rotation=45)
    ax[0].set_title("Numero di repository per dominio")
    ax[0].set_xlabel("Languages")
    ax[0].set_ylabel("Number")

stars = [
    (domain, df["Stars Count"].sum())
    for domain, df in datasets
]

stars = sorted(stars, key=lambda x: x[1], reverse=True)

for (domain, dataset) in stars:
    ax[1].bar(domain, dataset)
    ax[1].tick_params(axis='x', rotation=45)
    ax[1].set_title("Numero di stelle per dominio")
    ax[1].set_xlabel("Languages")
    ax[1].set_ylabel("Stars")
    
ax[1].ticklabel_format(style='plain', axis='y')

plt.show()

Quindi possiamo notare che il dataset contiene le 200 repository più popolari per ogni dominio.   
Il dominio più popolare è chiaramente Python, seguito da Go e Machine Learning

### Linguaggi più utilizzati in ogni dominio
Ogni repository ha un linguaggio predominante

Possiamo quindi trovare quali sono i linguaggi più usati in un certo dominio

In [ ]:
@interact
def languages(lang = ["all", *[x[0] for x in datasets]]):
    if lang == "all" or lang == None:
        fig, ax = plt.subplots(int(len(datasets) / 3), 3, figsize=(18, 20))
        plt.tight_layout(h_pad=5.0, pad=2)
        
        for i, (domain, dataset) in enumerate(datasets):
            grouped = (
                dataset
                    .dropna(subset=["Primary Language"])
                    .groupby("Primary Language")
                    .size()
                    .sort_values(ascending=False)
            )
        
            grouped = grouped.head(10)  # top 10 languages
            row = i // 3
            col = i % 3
        
            ax[row, col].bar(grouped.index, grouped.values)
        
            ax[row, col].set_title(domain)
            ax[row, col].set_ylabel("Numero di repository")
            ax[row, col].tick_params(axis='x', rotation=45)
    else:
        (domain, dataset) = next(filter(lambda x: x[0] == lang, datasets))
        grouped = dataset.dropna(subset=["Primary Language"]).groupby("Primary Language").size().sort_values(ascending=False)
        grouped = grouped.head(10)

        plt.figure(figsize=(10, 5))
        plt.bar(grouped.index, grouped.values)
        plt.title(domain)
        plt.ylabel("Numero di repository")
        plt.tick_params(axis='x', rotation=45)


Si può notare che in quasi tutti gli ambiti, il linguaggio più utilizzato è Python, però anche Go, che è un linguaggio relativamente recente, è molto usato.

## Dimensione della repository all'aumentare delle stelle
Forse è possibile trovare una correlazione tra numero di stelle di una repository e la sua dimensione, in quanto i progetti più longevi, e quindi con più stelle tendono ad essere più complessi

In [ ]:
for (domain, dataset) in datasets:
    # threshold = dataset["Size (KB)"].quantile(0.95)
    # filtered = dataset[dataset["Size (KB)"] <= threshold]
    plt.scatter(dataset["Stars Count"], dataset["Size (KB)"], alpha=0.3, label=domain)


plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Stars")
plt.ylabel("Size (KB)")
plt.title("Dimensione della repository a seconda delle stelle (scala logaritmica)")
plt.xscale("log")
plt.yscale("log")
plt.show()

Come si può notare dal grafico, in realtà non c'è una correlazione per cui il numero di stelle è correlato con la dimensione della repository. Infatti le repository con più stelle non sono quelle con la dimensione maggiore.

## Dimensione media della repository in base al dominio

In [ ]:
plt.figure(figsize=(11, 5))
for (domain, dataset) in sorted(datasets, key=lambda x: x[1]["Size (KB)"].mean(), reverse=True):
    plt.bar(domain, dataset["Size (KB)"].mean())

plt.tick_params(axis='x', rotation=45)
plt.title("Dimensione media in base al dominio")
plt.xlabel("Dominio")
plt.ylabel("Size (KB)")
plt.show()

Quindi il dominio che ha le repository di dimensione maggiore è C++

## Rateo fork rispetto a stelle

In [ ]:
@interact
def stars_to_forks(xscale = ["log", "linear"], yscale = ["log", "linear"]):
    plt.figure(figsize=(12,6))
    for (domain, dataset) in datasets:
        stars = dataset.sort_values(by="Stars Count")
        stars = stars.nlargest(100, "Stars Count")
        
        plt.scatter(stars["Stars Count"], stars["Forks Count"], label=domain)
    
    plt.xscale(xscale)
    plt.yscale(yscale)
    plt.title(f"Numero di fork rispetto alle stelle - Scala ${xscale}-${yscale}")
    
    plt.xlabel("Numero di stelle")
    plt.ylabel("Numero di fork")
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

Si può notare (più facilmente utilizzando la scala logaritmica) che il grafo ha una tendenza lineare, quindi più una repository è popolare, più tende ad avere fork

### Analisi issues aperti
Osserviamo la relazione tra il numero di issues aperti e il numero di stelle, il numero di issues aperti per dominio e il numero di issues aperti per ogni stella per dominio

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(25, 6))

sorted_dataset = full_dataset.sort_values(by="Stars Count")

ax[0].plot(sorted_dataset["Stars Count"], sorted_dataset["Open Issues Count"])

ax[0].set_title("Numero di issues aperti rispetto alle stelle")
ax[0].set_xlabel("Numero di stelle")
ax[0].set_ylabel("Numero di issues aperti") 

ax[1].set_title("Numero di issues aperti per dominio")
issues = [
    (domain, df["Open Issues Count"].sum())
    for domain, df in datasets
]

issues = sorted(issues, key=lambda x: x[1], reverse=True)


for (domain, dataset) in issues:
    ax[1].bar(domain, dataset)
    ax[1].tick_params(axis='x', rotation=45)
    ax[1].set_xlabel("Dominio")
    ax[1].set_ylabel("Issues")
    
ax[1].ticklabel_format(style='plain', axis='y')

plt.show()

plt.figure(figsize=(13,5))
issues = [
    (domain, df["Open Issues Count"].sum()/df["Stars Count"].sum())
    for domain, df in datasets
]

issues = sorted(issues, key=lambda x: x[1], reverse=True)
plt.title("Numero di issues aperti per ogni stella per dominio")
for (domain, dataset) in issues:
    plt.bar(domain, dataset)
    plt.tick_params(axis='x', rotation=45)
    plt.xlabel("Dominio")
    plt.ylabel("Issues")
    
plt.ticklabel_format(style='plain', axis='y')

plt.show()

Quindi in generale le repositories con più issues aperti sono C++ e Python, però il numero di issues aperti per persona (numero di stelle) è molto minore per python, mentre C++ rimane

### Percentuale delle repo con wiki, pages e projects

In [ ]:
features = ['Has Wiki', 'Has Pages', 'Has Projects']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(features):
    counts = full_dataset[feature].value_counts()
    
    labels = [str(val) for val in counts.index]
    values = counts.values
    
    axes[i].pie(
        values, 
        labels=labels, 
        autopct='%1.1f%%',  
        startangle=90,      
        colors=['#2ecc71', '#e74c3c']
    )
    
    axes[i].set_title(f'Presenza di {feature}')

plt.tight_layout()
plt.show()

Quindi le repo più popolari, molto spesso utilizzano queste feature di github. La feature più utilizzata è Github Projects, mentre quella meno utilizzata è Github Wiki